# Exploratory Data Analysis — LARS Dataset

## Table of Contents
1. [Dataset Overview](#dataset-overview)
2. [Image-Level Label Distributions](#image-level-label-distributions)
3. [Special Conditions](#special-conditions)
4. [Segmentation Categories](#segmentation-categories)
5. [Sample Visualisation](#sample-visualisation)
6. [Class Imbalance & Biases](#class-imbalance--biases)


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
import json
import os
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

DATA_ROOT = Path("../Data/lars_v1.0.0_annotations")
IMAGE_ROOT = Path("../Data/lars_v1.0.0_images")

SPLITS = ["train", "val", "test"]


## Dataset Overview

The LARS (Large Aquatic River and Sea) dataset is a maritime scene understanding dataset. It provides:
- **Image-level labels**: scene type, lighting, reflections, waves, and special conditions
- **Panoptic segmentation**: pixel-level masks with 11 categories (stuff + things)
- **Splits**: train / val / test (test has image-level labels only, no segmentation masks)


In [ ]:
# Load image-level annotations for all splits
annotations = {}
for split in SPLITS:
    with open(DATA_ROOT / split / "image_annotations.json") as f:
        annotations[split] = json.load(f)["annotations"]

for split, anns in annotations.items():
    print(f"{split}: {len(anns)} images")

# Quick look at a single annotation
print("\nExample annotation:")
print(json.dumps(annotations["train"][0], indent=2))


## Image-Level Label Distributions

Each image is annotated with four categorical labels: `scene_type`, `lighting`, `reflections`, and `waves`.


In [ ]:
all_anns = [a for split in SPLITS for a in annotations[split]]

label_fields = ["scene_type", "lighting", "reflections", "waves"]

fig, axes = plt.subplots(1, len(label_fields), figsize=(16, 4))

for ax, field in zip(axes, label_fields):
    counts = Counter(a["labels"][field] for a in all_anns)
    ax.bar(counts.keys(), counts.values())
    ax.set_title(field)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


## Special Conditions

Each image also has a set of boolean flags for challenging visual conditions (e.g. fog, glitter, rain).


In [ ]:
special_counts = Counter()
for a in all_anns:
    for flag, val in a["labels"]["special"].items():
        if val:
            special_counts[flag] += 1

flags, counts = zip(*special_counts.most_common())
plt.figure(figsize=(8, 4))
plt.bar(flags, counts)
plt.title("Special Condition Flags (True count across all images)")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## Segmentation Categories

The panoptic annotations contain 11 categories split into **stuff** (background regions: Water, Sky, Static Obstacle) and **things** (countable instances: Boat/ship, Buoy, Swimmer, etc.).


In [ ]:
# Load panoptic annotations (train + val only — test has none)
pan_anns = {}
for split in ["train", "val"]:
    with open(DATA_ROOT / split / "panoptic_annotations.json") as f:
        pan_anns[split] = json.load(f)

categories = {c["id"]: c for c in pan_anns["train"]["categories"]}

# Count segments per category
seg_counts = Counter()
for split in ["train", "val"]:
    for ann in pan_anns[split]["annotations"]:
        for seg in ann["segments_info"]:
            seg_counts[categories[seg["category_id"]]["name"]] += 1

names, counts = zip(*seg_counts.most_common())
colors = ["steelblue" if categories[[c["id"] for c in categories.values() if c["name"] == n][0]]["isthing"] else "coral"
          for n in names]

plt.figure(figsize=(10, 4))
plt.bar(names, counts, color=colors)
plt.title("Segment count per category (train + val)\nBlue = thing, Red = stuff")
plt.ylabel("Segments")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## Sample Visualisation

Visualise a few images alongside their panoptic masks and bounding boxes to get a feel for the data.


In [ ]:
# TODO: visualise N sample images with their panoptic mask overlaid and bounding boxes drawn
# Hint: panoptic masks are PNG files in panoptic_masks/, named after the annotation file_name field
N = 4


In [ ]:
# TODO: highlight key imbalances — e.g. ratio of dominant vs rare classes,
# split-level breakdowns, co-occurrence of special flags with rare scene types


## Class Imbalance & Biases

Summarise any notable imbalances in the dataset that could affect model training — e.g. dominance of `sea_like` scenes, scarcity of `night_like` images, or rare object categories like `Float`.
